## CSC 483 HW\#7 Intel Image Classification Competition (Fall 2026)

### Name: Jane Doe <== You should replace this with your name

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dropout
import matplotlib.pyplot as plt

### Load the training data. Split into training 80% and validation 20%.

In [ ]:
train_directory = '../input/csc-483-hw-7-fall-2026/train'
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory,
    labels='inferred',        # use names of subdirectories as target labels
    label_mode='categorical', # convert target class (int) to one-hot-vector
    validation_split=0.2,
    subset="training",
    seed=123,                 # use same random seed with valid_set
    class_names=None,
    color_mode='rgb',
    batch_size=32,
    image_size=(150, 150),
)

valid_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory,
    labels='inferred',
    label_mode='categorical',
    validation_split=0.2,
    subset="validation",
    seed=123,
    class_names=None,
    color_mode='rgb',
    batch_size=32,
    image_size=(150, 150),
)

In [ ]:
# Visualize some training examples.
plt.figure(figsize=(10, 12))
class_names = train_dataset.class_names
for images, labels in train_dataset.take(1):
    for i in range(30):
        ax = plt.subplot(6, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[np.argmax(labels[i])])
        plt.axis("off")

### Define a model.

In [ ]:
# A relatively simple CNN model
model = keras.Sequential()
model.add(keras.layers.Conv2D(64, (3,3), activation = 'relu', input_shape=(150,150,3)))
model.add(keras.layers.MaxPooling2D(2,2))

model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(128, activation='relu'))
model.add(Dropout(0.25))
model.add(keras.layers.Dense(6, activation='softmax'))

# Compile your model with your optimizer, loss, and metrics
opt = Adam(learning_rate = 0.001)  # set learning rate (same as default for now)
model.compile(optimizer=opt, loss='categorical_crossentropy', 
              metrics=['accuracy'])

In [ ]:
# Print model summary to ensure
model.summary()

In [ ]:
# Fit the model.  We train for 15 epochs for now.
history = model.fit(train_dataset, epochs=15, validation_data=valid_dataset)

### Visualize Training (accuracy and loss)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

### Examine the results for some validation examples.

In [ ]:
#Visualize 30 of the predicted data with green label for correct predictions
#and red label vise versa.
plt.figure(figsize=(15,15))
for images, labels in valid_dataset.take(1):
    for i in range(30):
        prediction = model.predict(images[i].numpy().reshape(1,150,150,3))
        plt.subplot(5, 6, i+1)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.grid(False)
        xlabel = plt.xlabel(class_names[prediction.argmax()])
        if (prediction.argmax() == labels[i].numpy().argmax()):
            xlabel.set_color("green")
        else:
            xlabel.set_color("red")
plt.show()

### Make Predictions for Test_Pred data.  
#### Note: Results are stored in a dictionary, which is keyed by file numbers (as integers).

In [ ]:
# Load the test_pred data (which has no target labels)
import os
import cv2

pred_directory = '../input/csc-483-hw-7-fall-2026/test_pred'
result_dict = {} # dictionary to store predictions (keyed by file number)

# iterate over files in that directory
for filename in os.listdir(pred_directory):
    f = os.path.join(pred_directory, filename)
    # checking if it is a file
    if os.path.isfile(f):
        fnum = int(filename[:-4]) # filename e.g. '103.jpg" -> 103
        img = cv2.imread(f)
        #img = img/255.0
        img = img.reshape(-1,150,150,3)
        pred = model.predict(img)
        result_dict[fnum] = pred[0]  # [0] because there is only one data
print (len(result_dict))

## (!!) Write the rest of the code by yourself.
### You must write the submission file, where the entries are sorted by the file number.